In [1]:
# 0.1 - load both Stage B tracks
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

_here     = Path.cwd()
REPO_ROOT = _here if (_here / 'Stage2_Outputs').exists() else _here.parent
IN_DIR    = REPO_ROOT / 'Stage2_Outputs'
DATA_DIR  = REPO_ROOT / 'Data Collection'

NAVY, BLUE, TEAL, AMBER = '#1F3864', '#2E75B6', '#17A589', '#E67E22'
RED, GREEN, GREY        = '#C0392B', '#27AE60', '#7F8C8D'
TEMPLATE = 'plotly_white'

ts_fcst = pd.read_csv(IN_DIR / 'TS_PD_forecasts_US_Q.csv', index_col=0, parse_dates=True)
ml_fcst = pd.read_csv(IN_DIR / 'ML_PD_forecasts_US_Q.csv', index_col=0, parse_dates=True)
ts_met  = pd.read_csv(IN_DIR / 'TS_metrics_US_Q.csv')
ml_met  = pd.read_csv(IN_DIR / 'ML_metrics_US_Q.csv')
ts_meta = pd.read_csv(IN_DIR / 'TS_run_metadata_US_Q.csv').set_index('key')['value']
ml_meta = pd.read_csv(IN_DIR / 'ML_run_metadata_US_Q.csv').set_index('key')['value']

# Model columns, excluding the OLS robustness variants and the SARIMAX interval bounds.
TS_MODELS = ['TS_Naive_RW', 'TS_AR1', 'TS_SARIMA', 'TS_SARIMAX', 'TS_ARDL', 'TS_VAR']
ML_MODELS = ['ML_OLS_primary', 'ML_Ridge', 'ML_Lasso', 'ML_Elastic_Net', 'ML_KRR',
             'ML_SVR', 'ML_Random_Forest', 'ML_XGBoost', 'ML_MLP']

# Historical target for context in Figure 1.1; the comparison itself does not need it.
try:
    hist = (pd.read_csv(DATA_DIR / 'Stage1_final_regressors_US_Q.csv',
                        index_col=0, parse_dates=True)['us_delinquency_rate_raw'].dropna())
except Exception as e:
    hist = None
    print(f'History unavailable ({type(e).__name__}); Figure 1.1 will show the horizon only.')

assert ts_fcst.index.equals(ml_fcst.index), 'forecast indices differ'
print(f'Loaded from {IN_DIR}')
print(f'  horizon      {ts_fcst.index.min().date()} to {ts_fcst.index.max().date()}, '
      f'{len(ts_fcst)} quarters')
print(f'  TS models    {len(TS_MODELS)}   ML models {len(ML_MODELS)}')

Loaded from c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Stage2_Outputs
  horizon      2026-03-31 to 2030-12-31, 20 quarters
  TS models    6   ML models 9


In [2]:
# 0.2 - Table 0.1 compatibility check
# Everything downstream is gated on this. The two tracks were built independently and
# agree on some axes and not others; a comparison is only legitimate on the axes that
# match, and the rest have to be reported as differences rather than silently averaged.
def _g(s, *keys):
    for k in keys:
        if k in s.index:
            return str(s[k])
    return '—'

ml_n = int(re.search(r'(\d+)Q', 'Clean (14Q)').group(1))
ml_n = int(re.search(r'\((\d+)Q\)', ml_met['Window'].unique()[1]).group(1))

axes = [
    ('Target series',     _g(ts_meta, 'target'),         _g(ml_meta, 'target')),
    ('Scoring window',    _g(ts_meta, 'clean_window'),   _g(ml_meta, 'clean_window')),
    ('Scored quarters',   _g(ts_meta, 'holdout_n'),      str(ml_n)),
    ('Forecast horizon',  _g(ts_meta, 'horizon'),        str(len(ml_fcst))),
    ('Target variant',    _g(ts_meta, 'target_variant'), _g(ml_meta, 'target_variant')),
    ('Forecast protocol', _g(ts_meta, 'protocol'),       _g(ml_meta, 'protocol')),
    ('Regressors',        _g(ts_meta, 'regressors'),     _g(ml_meta, 'features')),
]
chk = pd.DataFrame(axes, columns=['Axis', 'TS track (03a)', 'ML track (03b)'])
chk['Match'] = np.where(chk['TS track (03a)'] == chk['ML track (03b)'], 'Yes', 'No')

print('Table 0.1 - Compatibility of the two Stage B tracks\n')
for _, r in chk.iterrows():
    print(f"{r['Axis']:<18s} [{r['Match']:>3s}]")
    print(f"    TS  {r['TS track (03a)']}")
    print(f"    ML  {r['ML track (03b)']}")

n_match = int((chk['Match'] == 'Yes').sum())
print(f'\n{n_match} of {len(chk)} axes match.')
print('Legitimate: both tracks are scored on the same quarters, against the same '
      'observed target values,\nover the same 20-quarter horizon. 03b confirms the '
      'spline and raw targets are identical on\nthis window, so the target-variant '
      'difference affects training only, never the evaluation.')
print('Not legitimate: RMSE levels across tracks. The protocols differ in information '
      'set, so each\ntrack is compared against its own naive baseline via skill, not '
      'against the other on RMSE.')

Table 0.1 - Compatibility of the two Stage B tracks

Target series      [Yes]
    TS  us_delinquency_rate
    ML  us_delinquency_rate
Scoring window     [Yes]
    TS  2022-09-30 to 2025-12-31
    ML  2022-09-30 to 2025-12-31
Scored quarters    [Yes]
    TS  14
    ML  14
Forecast horizon   [Yes]
    TS  20
    ML  20
Target variant     [ No]
    TS  raw, COVID quarters masked rather than splined
    ML  spline-adjusted, covid_dummy in specification
Forecast protocol  [ No]
    TS  expanding-window refit, h=1, realised target at every origin
    ML  single origin, h=1..20, no refit, no realised target after origin
Regressors         [ No]
    TS  us_house_price_yoy_L3 us_consumer_confidence_L2 us_unemployment_L0 us_credit_qoq_growth_L6 us_gdp_yoy_growth_L2 us_cpi_L6
    ML  us_credit_qoq_growth_L6 us_unemployment_L0 us_house_price_yoy_L3 us_gdp_yoy_growth_L2 us_consumer_confidence_L2 us_cpi_L0 us_indprod_yoy_L3 covid_dummy

4 of 7 axes match.
Legitimate: both tracks are scored on the sa

In [3]:
# 1.1 - Figure 1.1 forecast paths, both tracks
fig = go.Figure()
if hist is not None:
    h = hist[hist.index >= '2015-01-01']
    fig.add_trace(go.Scatter(x=h.index, y=h.values, mode='lines', name='Actual',
                             line=dict(color=NAVY, width=2.6)))
fig.add_trace(go.Scatter(
    x=list(ts_fcst.index) + list(ts_fcst.index[::-1]),
    y=list(ts_fcst['TS_SARIMAX_hi95']) + list(ts_fcst['TS_SARIMAX_lo95'][::-1]),
    fill='toself', fillcolor='rgba(127,140,141,0.13)', line=dict(width=0),
    hoverinfo='skip', name='SARIMAX 95% (macro treated as known)'))
for c in TS_MODELS:
    fig.add_trace(go.Scatter(x=ts_fcst.index, y=ts_fcst[c], mode='lines',
                             name=c.replace('TS_', '') + ' (TS)',
                             line=dict(color=TEAL, width=1.3,
                                       dash='dot' if 'Naive' in c else 'solid'),
                             opacity=0.75, legendgroup='TS'))
for c in ML_MODELS:
    fig.add_trace(go.Scatter(x=ml_fcst.index, y=ml_fcst[c], mode='lines',
                             name=c.replace('ML_', '') + ' (ML)',
                             line=dict(color=AMBER, width=1.3), opacity=0.75,
                             legendgroup='ML'))
fig.add_trace(go.Scatter(x=ts_fcst.index, y=ts_fcst['TS_SARIMAX'], mode='lines',
                         name='SARIMAX (TS deliverable)',
                         line=dict(color=GREEN, width=3.0)))
fig.add_trace(go.Scatter(x=ml_fcst.index, y=ml_fcst['ML_OLS_primary'], mode='lines',
                         name='OLS (ML deliverable)', line=dict(color=RED, width=3.0)))
fig.update_layout(
    title=dict(text='<b>Figure 1.1 - Stage B forecast paths, both tracks</b>'
                    f'<br><span style="font-size:11.5px;color:{GREY}">'
                    'Teal = time-series track, amber = machine-learning track, '
                    'bold = each track\'s deliverable</span>',
               font=dict(size=15, color=NAVY), x=0.015, xanchor='left'),
    template=TEMPLATE, height=520, xaxis_title='Quarter',
    yaxis_title='Delinquency rate (%)',
    legend=dict(font=dict(size=9)), margin=dict(t=95, b=45))
fig.show()

In [4]:
# 1.2 - Table 1.1 path divergence
# The paths are directly comparable: same index, same units, no protocol caveat.
rows = []
for lbl, cols, d in [('TS (03a)', TS_MODELS, ts_fcst), ('ML (03b)', ML_MODELS, ml_fcst)]:
    s = d[cols]
    rows.append({'Track': lbl, 'Models': len(cols),
                 '2026 Q1 mean': s.iloc[0].mean(), '2030 Q4 mean': s.iloc[-1].mean(),
                 '2030 Q4 min': s.iloc[-1].min(), '2030 Q4 max': s.iloc[-1].max(),
                 'Within-track spread': s.iloc[-1].max() - s.iloc[-1].min()})
print('Table 1.1 - Forecast path summary, 2026 Q1 to 2030 Q4\n')
print(pd.DataFrame(rows).round(3).to_string(index=False))

gap = ml_fcst[ML_MODELS].mean(axis=1) - ts_fcst[TS_MODELS].mean(axis=1)
dlv = ml_fcst['ML_OLS_primary'] - ts_fcst['TS_SARIMAX']
print(f'\nTrack ensemble means, ML minus TS: start {gap.iloc[0]:+.3f}pp, '
      f'end {gap.iloc[-1]:+.3f}pp, mean {gap.mean():+.3f}pp, largest {gap.abs().max():.3f}pp')
print(f'Deliverables, OLS minus SARIMAX:  mean {dlv.mean():+.3f}pp, '
      f'largest {dlv.abs().max():.3f}pp')

inband = ((ml_fcst['ML_OLS_primary'] >= ts_fcst['TS_SARIMAX_lo95']) &
          (ml_fcst['ML_OLS_primary'] <= ts_fcst['TS_SARIMAX_hi95']))
print(f'OLS path inside the SARIMAX 95% band: {int(inband.sum())} of {len(inband)} quarters')

step_ml = ml_fcst['ML_OLS_primary'].diff().abs().max()
step_ts = ts_fcst['TS_SARIMAX'].diff().abs().max()
print(f'\nLargest single-quarter move: OLS {step_ml:.3f}pp, SARIMAX {step_ts:.3f}pp. '
      f'The OLS movement is the\n2026 Q2 dip traced in 03b to the credit-growth regressor '
      f'turning negative at lag six.')

Table 1.1 - Forecast path summary, 2026 Q1 to 2030 Q4

   Track  Models  2026 Q1 mean  2030 Q4 mean  2030 Q4 min  2030 Q4 max  Within-track spread
TS (03a)       6         2.917         2.834        2.309        3.345                1.036
ML (03b)       9         3.292         3.152        2.772        3.277                0.506

Track ensemble means, ML minus TS: start +0.374pp, end +0.318pp, mean +0.541pp, largest 0.805pp
Deliverables, OLS minus SARIMAX:  mean +0.301pp, largest 0.651pp
OLS path inside the SARIMAX 95% band: 19 of 20 quarters

Largest single-quarter move: OLS 0.917pp, SARIMAX 0.037pp. The OLS movement is the
2026 Q2 dip traced in 03b to the credit-growth regressor turning negative at lag six.


In [11]:
# 2.1 - Table 2.1 harmonised accuracy on the common window
# TS uses the static-protocol variant: parameters estimated once before the window and
# frozen, which is the protocol the ML track follows. 03a's Table D shows this costs the
# TS models almost nothing, so the remaining difference between tracks is the information
# set, not re-estimation.
TS_BM = ['Naive RW']
ML_BM = ['Naive persistence', 'Long-run mean', 'Unemployment only']

t = ts_met[ts_met['window'] == 'ml_holdout_static'][['model', 'RMSE', 'MAE', 'skill%']].copy()
t.columns = ['Model', 'RMSE', 'MAE', 'Skill %']
t['Track'] = np.where(t['Model'].isin(TS_BM), 'TS benchmark', 'TS')

m = ml_met[ml_met['Window'] == 'Clean (14Q)'][
        ['Model', 'RMSE', 'MAE', 'Skill vs persist']].copy()
m.columns = ['Model', 'RMSE', 'MAE', 'Skill %']
m['Skill %'] = m['Skill %'] * 100          # 03b stores a fraction, 03a stores percent
m['Track'] = np.where(m['Model'].isin(ML_BM), 'ML benchmark', 'ML')

comp = (pd.concat([t, m], ignore_index=True)
          .sort_values('Skill %', ascending=False).reset_index(drop=True))
IS_BM = comp['Track'].str.contains('benchmark').to_numpy()
comp['Rank']  = np.where(IS_BM, '', (~IS_BM).cumsum().astype(str))
comp['Track'] = comp['Track'].str.replace(' benchmark', '', regex=False)
tbl = comp[['Rank', 'Track', 'Model', 'RMSE', 'MAE', 'Skill %']]

# Benchmarks are italicised rather than removed: they are the zero line each track is
# measured against, not entrants in the ranking.
def _row(r):
    return ['font-style:italic'] * len(r) if IS_BM[r.name] else [''] * len(r)

display(tbl.style
    .apply(_row, axis=1)
    .format({'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'Skill %': '{:+.1f}'})
    .set_properties(subset=['Track', 'Model'], **{'text-align': 'left'})
    .set_caption('Table 2.1 — Accuracy on the common window, 2022 Q3 to 2025 Q4 '
                 '(n = 14 for both tracks)')
    .set_table_styles([
        {'selector': 'caption',
         'props': 'caption-side:top; text-align:left; font-size:13px; '
                  'font-weight:600; padding-bottom:10px'},
        {'selector': 'thead th',
         'props': 'border-top:1.5px solid; border-bottom:1px solid; background:none; '
                  'font-weight:600; text-align:right; padding:6px 16px'},
        {'selector': 'td', 'props': 'text-align:right; padding:5px 16px; border:none'},
        {'selector': 'tbody tr:last-child td', 'props': 'border-bottom:1.5px solid'},
        {'selector': '', 'props': 'border-collapse:collapse; font-size:12.5px'}])
    .hide(axis='index'))

ts_naive = float(t.loc[t['Model'] == 'Naive RW', 'RMSE'].iloc[0])
ml_naive = float(m.loc[m['Model'] == 'Naive persistence', 'RMSE'].iloc[0])
n_beat_ts = int((t[~t['Model'].isin(TS_BM)]['Skill %'] > 0).sum())
n_beat_ml = int((m[~m['Model'].isin(ML_BM)]['Skill %'] > 0).sum())
u_skill = float(m.loc[m['Model'] == 'Unemployment only', 'Skill %'].iloc[0])
n_below = int((m[~m['Model'].isin(ML_BM)]['Skill %'] < u_skill).sum())

print('Italicised rows are benchmarks. Skill is measured against each track\'s own naive '
      'baseline and is\ncomparable across tracks. RMSE is not.')
print(f'\nNaive baseline on the same 14 quarters: TS {ts_naive:.4f}, ML {ml_naive:.4f}, '
      f'a factor of {ml_naive / ts_naive:.2f}.')
print('Both are a random walk on the same observed values, so the gap is entirely the '
      'information set:\nthe TS baseline carries forward last quarter\'s realised value, '
      'the ML baseline carries forward\n2020 Q4 for seven to twenty quarters. Comparing '
      'RMSE across tracks would measure this, not the models.')
print(f'\nBeating their own naive baseline: TS {n_beat_ts} of 5, ML {n_beat_ml} of 9.')
print(f'The single-regressor "Unemployment only" benchmark ({u_skill:+.1f}%) outranks '
      f'{n_below} of the 9 ML models.')

Rank,Track,Model,RMSE,MAE,Skill %
1,TS,SARIMAX,0.1331,0.0844,+34.0
2,TS,ARDL,0.1511,0.0977,+25.1
3,TS,SARIMA,0.1580,0.0942,+21.7
4,TS,AR(1),0.1737,0.1305,+13.9
5,ML,MLP,0.6464,0.5079,+8.7
,TS,Naive RW,0.2018,0.1436,+0.0
,ML,Naive persistence,0.7080,0.6451,+0.0
,ML,Unemployment only,0.7417,0.6746,-4.8
6,ML,KRR,0.7647,0.6155,-8.0
7,TS,VAR,0.2215,0.1224,-9.7


Italicised rows are benchmarks. Skill is measured against each track's own naive baseline and is
comparable across tracks. RMSE is not.

Naive baseline on the same 14 quarters: TS 0.2018, ML 0.7080, a factor of 3.51.
Both are a random walk on the same observed values, so the gap is entirely the information set:
the TS baseline carries forward last quarter's realised value, the ML baseline carries forward
2020 Q4 for seven to twenty quarters. Comparing RMSE across tracks would measure this, not the models.

Beating their own naive baseline: TS 4 of 5, ML 1 of 9.
The single-regressor "Unemployment only" benchmark (-4.8%) outranks 8 of the 9 ML models.


In [6]:
# 2.2 - Figure 2.1 RMSE on separate scales, why the levels cannot be pooled
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                    subplot_titles=('Time-series track (03a)',
                                    'Machine-learning track (03b)'))
for col, (d, bmlist, colr) in enumerate(
        [(t, TS_BM, TEAL), (m, ML_BM, AMBER)], start=1):
    d2 = d.sort_values('RMSE')
    fig.add_trace(go.Bar(
        x=d2['RMSE'], y=d2['Model'], orientation='h', showlegend=False,
        marker_color=[GREY if mm in bmlist else colr for mm in d2['Model']],
        text=d2['RMSE'].round(3), textposition='outside'), row=1, col=col)
    base = float(d2.loc[d2['Model'].isin(bmlist), 'RMSE'].min())
    fig.add_vline(x=base, line=dict(color=RED, width=1.6, dash='dash'), row=1, col=col)
fig.update_layout(
    title=dict(text='<b>Figure 2.1 - Test RMSE, same 14 quarters, separate scales</b>'
                    f'<br><span style="font-size:11.5px;color:{GREY}">'
                    'Grey = benchmark, dashed red = naive baseline. The axes differ by '
                    'roughly 3.5x: that is the protocol, not the models</span>',
               font=dict(size=15, color=NAVY), x=0.015, xanchor='left'),
    template=TEMPLATE, height=430, margin=dict(t=105, b=45, l=130))
fig.update_xaxes(title_text='RMSE (pp)')
fig.show()

In [7]:
# 2.3 - Figure 2.2 skill against each track's own naive baseline
d = comp[~comp['Track'].str.contains('benchmark')].sort_values('Skill %')
fig = go.Figure(go.Bar(
    x=d['Skill %'], y=d['Model'] + '  (' + d['Track'] + ')', orientation='h',
    marker_color=[TEAL if tr == 'TS' else AMBER for tr in d['Track']],
    text=d['Skill %'].round(1), textposition='outside', showlegend=False))
fig.add_vline(x=0, line=dict(color=RED, width=2))
fig.update_layout(
    title=dict(text='<b>Figure 2.2 - Skill against each track\'s own naive baseline</b>'
                    f'<br><span style="font-size:11.5px;color:{GREY}">'
                    'Zero is the naive forecast under that track\'s protocol. This is the '
                    'cross-track comparable quantity</span>',
               font=dict(size=15, color=NAVY), x=0.015, xanchor='left'),
    template=TEMPLATE, height=520, xaxis_title='Skill vs naive (%)',
    margin=dict(t=105, b=45, l=190))
fig.show()

In [8]:
# 3.1 - Table 3.1 interpretability and reproducibility
# Neither axis is measurable on the test window, so both are assessed from evidence the
# two notebooks already produced rather than asserted.
rows = [
    ('Coefficients an auditor can read',
     'SARIMAX, ARDL, AR(1): yes. SARIMA: no macro terms. VAR: 25-cell system.',
     'OLS, Ridge, Lasso, Elastic Net: yes. KRR, SVR, RF, XGBoost, MLP: no.'),
    ('Attribution available',
     'ARDL and SARIMAX coefficients are directly readable.',
     'SHAP computed for OLS and the two leading challengers. KRR reproduces the OLS '
     'ordering almost exactly (Spearman 0.976); MLP departs (0.762).'),
    ('Uncertainty interval',
     'SARIMAX 95% band, though it treats the Stage A macro paths as known and is '
     'therefore a lower bound.',
     'None produced. Point forecasts only.'),
    ('Reproducible on re-run',
     'No. auto_arima order search is unseeded; SARIMA holdout RMSE moved 0.1523 to '
     '0.1577 on a re-run with identical inputs.',
     'Yes. SEED = 42 throughout; the run reproduced exactly.'),
    ('Stable to design choices',
     'Yes on re-estimation: Table D shows freezing parameters changes RMSE by at most '
     '0.018.',
     'No. Switching the tuning criterion from MAE to RMSE moves KRR test RMSE by 20%.'),
    ('Rank stability across windows',
     'Poor. SARIMAX is 4th of 6 on the n=67 backtest and 1st on the n=14 holdout.',
     'Not testable. A single origin gives one window.'),
    ('Statistical power',
     'n=67 on the full backtest supports inference; the n=14 holdout does not.',
     'n=14. No model differs significantly from persistence (all DM p > 0.24).'),
]
print('Table 3.1 - Interpretability and reproducibility\n')
for a, b, c in rows:
    print(f'{a}')
    print(f'    TS  {b}')
    print(f'    ML  {c}\n')

print('Sign convention warning: 03a reports DM with positive meaning the model beats the '
      'benchmark;\n03b reports positive meaning it loses. The two DM columns are not '
      'directly comparable and are\nnot tabled together anywhere in this notebook.')

Table 3.1 - Interpretability and reproducibility

Coefficients an auditor can read
    TS  SARIMAX, ARDL, AR(1): yes. SARIMA: no macro terms. VAR: 25-cell system.
    ML  OLS, Ridge, Lasso, Elastic Net: yes. KRR, SVR, RF, XGBoost, MLP: no.

Attribution available
    TS  ARDL and SARIMAX coefficients are directly readable.
    ML  SHAP computed for OLS and the two leading challengers. KRR reproduces the OLS ordering almost exactly (Spearman 0.976); MLP departs (0.762).

Uncertainty interval
    TS  SARIMAX 95% band, though it treats the Stage A macro paths as known and is therefore a lower bound.
    ML  None produced. Point forecasts only.

Reproducible on re-run
    TS  No. auto_arima order search is unseeded; SARIMA holdout RMSE moved 0.1523 to 0.1577 on a re-run with identical inputs.
    ML  Yes. SEED = 42 throughout; the run reproduced exactly.

Stable to design choices
    TS  Yes on re-estimation: Table D shows freezing parameters changes RMSE by at most 0.018.
    ML  No. Switc

## Best-performing models and what each is good for

No overall winner is declared. The two tracks answer different questions, and the
protocol that makes one look accurate is the same protocol that makes it less useful
for the deliverable.

### Time-series track

**SARIMAX — 34.0% skill, best on the window.** Six lagged macro regressors with readable
coefficients, and the only model in either track producing an uncertainty interval.
Against it: 4th of 6 on the n=67 backtest, so its holdout win rests on fourteen quarters
of one regime; the order search is unseeded; and the 95% band treats Stage A's projected
macro paths as known, so it understates true uncertainty.

**ARDL — 25.1% skill, the most stable.** First on the full n=67 backtest and second on
the holdout, the only model without a large rank change between windows. Hand-rolled
least squares in lag form, so it is fully transparent. Against it: no interval, and it
produces the highest 2030 Q4 endpoint of any model in either track at 3.35%, extrapolating
the recent rise rather than mean-reverting.

**SARIMA — 21.7% skill, no macro inputs.** Immune to Stage A forecast error, since it
consumes none. That is also its limitation: a model with no macroeconomic conditioning
cannot answer the IFRS 9 question, which requires PD conditional on economic scenarios.
It is a floor on what the macro regressors need to beat, not a candidate deliverable.

### Machine-learning track

**MLP — +8.7% skill, the only model in the track beating its baseline.** Against it:
the margin is not significant (DM p = 0.79 on n=14); validation error below training
error indicates it is underfitting the unusually flat 2017–2020 window rather than
generalising; and SHAP shows it departs furthest from the linear structure, assigning
GDP growth almost no weight where OLS assigns 11%. It is a black box on fourteen test
quarters.

**KRR — −8.0% skill, most useful as a diagnostic.** It loses to persistence, but its
SHAP ordering agrees with OLS at Spearman 0.976, meaning a flexible nonlinear method
recovers essentially the linear structure. That is evidence the relationship is linear,
which supports OLS as the deliverable. Against it: the 20% swing in test RMSE from the
tuning criterion alone makes any single number unreliable.

**XGBoost — −28.8% skill, third by rank but not a genuine challenger.** Training RMSE
of 0.0095 against validation RMSE of 1.0072 is near-total overfitting on 117 training
observations. It is included for completeness and as the clearest illustration of why
tree ensembles need more data than a quarterly macro panel provides.

### What the comparison establishes

Four of five time-series models beat their baseline; eight of nine machine-learning
models do not, and a single-regressor unemployment benchmark outranks eight of them.
But the time-series protocol forecasts one quarter ahead from a realised previous
observation, while the machine-learning protocol projects twenty quarters from a single
origin. The first is a monitoring question; the second is the IFRS 9 lifetime question
and the one the deliverable has to answer. The ranking and the relevance run in opposite
directions, so neither track dominates.

Within-track disagreement runs the other way from what the accuracy tables suggest. The
six time-series models span 1.04pp at 2030 Q4; the nine machine-learning models span
0.51pp. The track that scores worse is the one whose models agree more with each other.

The two deliverables agree more than the accuracy gap implies. OLS and SARIMAX differ by
0.30pp on average across the horizon and the OLS path sits inside the SARIMAX 95% interval
in 19 of 20 quarters. The exception is 2026 Q2, where OLS drops 0.92pp on the credit-growth
regressor turning negative at lag six while SARIMAX moves 0.04pp. That single quarter is
the only point on the horizon where the two tracks genuinely disagree. Both otherwise
point to a delinquency rate near 3% through 2030, with the machine-learning paths roughly
half a point higher throughout.